In [1]:
%%writefile /kaggle/working/08_train_student.py
import os, time, math, random, json, argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.transforms.v2 as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, KFold

CFG = dict(
    backbone="vit_small_patch14_dinov2.lvd142m",
    tag="dinov2s",
    epochs=6,
    batch=4,            # studi per batch (data v2: pakai 2 karena 5 view x 256 px)
    lr_backbone=2e-5,   # untuk ViT dengan llrd < 1: ini LR blok TERATAS
    lr_head=1e-3,
    llrd=1.0,           # layer-wise LR decay; 1.0 = tidak dipakai (CNN)
    wd=1e-2,
    warmup_frac=0.05,
    val_frac=0.10,
    seed=42,
    workers=2,          # 2 per proses, karena 2 proses berjalan bersamaan di 4 CPU
    fold=-1,            # -1 = split 90/10 lama; 0..nfolds-1 = mode K-fold
    nfolds=5,
    alpha=1.0,          # porsi label laporan dalam label training (1.0 = tanpa guru)
    oof="oof_effb0_kf.csv",
    meta="_meta.csv",   # "_meta.csv" = data v1, "_meta_v2.csv" = data v2
    n_views=3,          # diisi otomatis dari data
    debug=False,
)
COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT = "/kaggle/working"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]


# ---------------- util ----------------
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def find_file(name):
    """Cari file di /kaggle/input tanpa masuk ke folder kompetisi (isinya ratusan ribu DICOM)."""
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(f"{name} tidak ditemukan di /kaggle/input")


def mean_auc(y, p):
    """AUC per kelas; label 0.5 (uncertain) dan kosong diabaikan."""
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk, pk = y[ok, k], p[ok, k]
        aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
    return float(np.nanmean(aucs)), aucs


# ---------------- data ----------------
class KneeDS(Dataset):
    def __init__(self, uids, ymat, vol_dir, train):
        self.uids, self.y, self.vol_dir, self.train = uids, ymat, vol_dir, train
        # Tanpa flip kiri-kanan: flip bisa menukar sisi medial/lateral
        self.affine = T.RandomAffine(degrees=10, translate=(0.06, 0.06), scale=(0.9, 1.1))

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        d = np.load(os.path.join(self.vol_dir, f"{self.uids[i]}.npz"))
        vol = torch.from_numpy(d["vol"]).float() / 255.0            # (V, D, H, W)
        vmask = torch.from_numpy(d["mask"]).float()                 # (V,)
        if self.train:
            views = []
            for v in range(vol.shape[0]):
                x = self.affine(vol[v])                             # transformasi sama untuk semua slice
                x = x * random.uniform(0.85, 1.15) + random.uniform(-0.08, 0.08)
                views.append(x.clamp(0, 1))
            vol = torch.stack(views)
        y = torch.tensor(self.y[i], dtype=torch.float32)
        ym = (~torch.isnan(y)).float()
        return vol, vmask, torch.nan_to_num(y, 0.0), ym


# ---------------- model ----------------
def backbone_kwargs(name):
    """DINOv2 di timm defaultnya 518 px; kita pakai 224 (positional embedding di-resample otomatis)."""
    if "dinov2" in name:
        return dict(img_size=224, drop_path_rate=0.05)
    return {}


class KneeNet(nn.Module):
    def __init__(self, backbone, n_views=3, n_out=len(LABELS), pretrained=True):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0,
                                     in_chans=3, **backbone_kwargs(backbone))
        f = self.enc.num_features
        self.att = nn.ModuleList([nn.Sequential(nn.Linear(f, 128), nn.Tanh(), nn.Linear(128, 1))
                                  for _ in range(n_views)])
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(f * n_views, n_out))

    def forward(self, vol, vmask, return_att=False):
        B, V, D, H, W = vol.shape
        prev = torch.cat([vol[:, :, :1], vol[:, :, :-1]], 2)
        nxt = torch.cat([vol[:, :, 1:], vol[:, :, -1:]], 2)
        x = torch.stack([prev, vol, nxt], 3)                        # (B, V, D, 3, H, W)  2.5D
        x = (x - 0.45) / 0.225
        feats = self.enc(x.flatten(0, 2)).view(B, V, D, -1)         # (B, V, D, F)
        pooled, atts = [], []
        for v in range(V):
            a = self.att[v](feats[:, v]).softmax(1)                 # (B, D, 1) bobot per slice
            pooled.append((a * feats[:, v]).sum(1))
            atts.append(a.squeeze(-1))
        z = torch.stack(pooled, 1) * vmask[:, :, None]              # view yang hilang dinolkan
        logits = self.head(z.flatten(1))
        return (logits, torch.stack(atts, 1)) if return_att else logits


def param_groups(net):
    """Head: lr_head. Backbone ViT + llrd<1: LR mengecil per blok dari atas ke bawah."""
    groups = [{"params": list(net.att.parameters()) + list(net.head.parameters()),
               "lr": CFG["lr_head"]}]
    enc = net.enc
    if CFG["llrd"] < 1.0 and hasattr(enc, "blocks"):
        L = len(enc.blocks)
        layers = {i: [] for i in range(L + 2)}
        for n, p in enc.named_parameters():
            if n.startswith("blocks."):
                lid = int(n.split(".")[1]) + 1
            elif n.startswith(("patch_embed", "cls_token", "pos_embed", "reg_token", "mask_token")):
                lid = 0
            else:                                   # norm akhir dll.
                lid = L + 1
            layers[lid].append(p)
        for lid, ps in layers.items():
            if ps:
                groups.append({"params": ps,
                               "lr": CFG["lr_backbone"] * CFG["llrd"] ** (L + 1 - lid)})
        lrs = [g["lr"] for g in groups[1:]]
        print(f"LLRD {CFG['llrd']}: LR backbone {min(lrs):.2e} (bawah) .. {max(lrs):.2e} (atas)")
    else:
        groups.append({"params": list(enc.parameters()), "lr": CFG["lr_backbone"]})
    return groups


def masked_bce(logits, y, ym):
    loss = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
    return (loss * ym).sum() / ym.sum().clamp(min=1)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    ps = []
    for vol, vmask, _, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            ps.append(torch.sigmoid(model(vol.cuda(non_blocking=True),
                                          vmask.cuda(non_blocking=True)).float()).cpu())
    return torch.cat(ps).numpy()


# ---------------- main ----------------
def main():
    seed_all(CFG["seed"])
    vol_dir = os.path.dirname(find_file(CFG["meta"]))
    lab_path = find_file("report_labels.csv")
    print("Volume :", vol_dir)
    print("Label  :", lab_path)

    train_csv = pd.read_csv(os.path.join(COMP, "train.csv"))
    gold = train_csv[train_csv[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]
    gold_ids = set(gold["StudyInstanceUID"])
    have = {f[:-4] for f in os.listdir(vol_dir) if f.endswith(".npz")}

    rep = pd.read_csv(lab_path)
    rep = rep[rep["StudyInstanceUID"].isin(have) & ~rep["StudyInstanceUID"].isin(gold_ids)]
    gold = gold[gold["StudyInstanceUID"].isin(have)].reset_index(drop=True)
    # ---- label hasil perbaikan: alpha * laporan + (1 - alpha) * guru OOF ----
    rep_ref = rep.copy()
    if CFG["alpha"] < 1.0:
        oof = rep[["StudyInstanceUID"]].merge(pd.read_csv(find_file(CFG["oof"])),
                                              on="StudyInstanceUID", how="left")
        Tm = oof[LABELS].values.astype(float)
        assert not np.isnan(Tm).any(), "ada studi tanpa prediksi OOF guru"
        R = rep[LABELS].values.astype(float)
        rep_ref[LABELS] = np.where(np.isnan(R), Tm, CFG["alpha"] * R + (1 - CFG["alpha"]) * Tm)
        print(f"Label diperbaiki: alpha={CFG['alpha']} (guru: {CFG['oof']})")

    # split train/val SELALU sama (42) untuk semua model
    if CFG["fold"] >= 0:
        kf = KFold(n_splits=CFG["nfolds"], shuffle=True, random_state=42)
        tr_idx, va_idx = list(kf.split(rep))[CFG["fold"]]
        tr_i, va_i = rep.index[tr_idx], rep.index[va_idx]
        CFG["tag"] = f"{CFG['tag']}_f{CFG['fold']}"
    else:
        tr0, va0 = train_test_split(rep, test_size=CFG["val_frac"], random_state=42)
        tr_i, va_i = tr0.index, va0.index
    tr = rep_ref.loc[tr_i]                    # target training: label hasil perbaikan
    va = rep.loc[va_i]                        # val vs label laporan asli (sebanding dgn run lama)
    va_ref = rep_ref.loc[va_i]                # val vs label perbaikan (dibinerkan di 0.5)
    if CFG["debug"]:
        tr, va, va_ref, CFG["epochs"] = tr.head(64), va.head(32), va_ref.head(32), 1
    y_va_ref = va_ref[LABELS].values.astype(np.float32)
    y_va_ref = np.where(np.isnan(y_va_ref), np.nan, (y_va_ref >= 0.5).astype(np.float32))
    print(f"Train {len(tr)} | Val {len(va)} | Gold holdout {len(gold)}")

    def loader(df, train):
        ds = KneeDS(df["StudyInstanceUID"].tolist(), df[LABELS].values.astype(np.float32),
                    vol_dir, train)
        return DataLoader(ds, batch_size=CFG["batch"], shuffle=train, drop_last=train,
                          num_workers=CFG["workers"], pin_memory=True,
                          persistent_workers=CFG["workers"] > 0)

    dl_tr, dl_va, dl_gold = loader(tr, True), loader(va, False), loader(gold, False)

    # 1 GPU saja: DataParallel di Kaggle T4 x2 (tanpa P2P) ~17x lebih lambat
    torch.backends.cudnn.benchmark = True
    CFG["n_views"] = int(np.load(os.path.join(vol_dir, f"{tr['StudyInstanceUID'].iloc[0]}.npz"))["mask"].shape[0])
    print(f"Jumlah view per studi: {CFG['n_views']}")
    net = KneeNet(CFG["backbone"], n_views=CFG["n_views"]).cuda()
    if "vit" not in CFG["backbone"]:
        net = net.to(memory_format=torch.channels_last)
    model = net
    tag = CFG["tag"]
    print(f"Backbone: {CFG['backbone']} | tag: {tag} | "
          f"parameter: {sum(p.numel() for p in net.parameters())/1e6:.1f} juta")
    with open(os.path.join(OUT, f"{tag}_cfg.json"), "w") as f:
        json.dump(CFG, f, indent=2)
    opt = torch.optim.AdamW(param_groups(net), weight_decay=CFG["wd"])
    total = CFG["epochs"] * len(dl_tr)
    warm = max(1, int(total * CFG["warmup_frac"]))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda")

    best, log = -1.0, []
    for ep in range(CFG["epochs"]):
        model.train()
        t0, run = time.time(), 0.0
        for it, (vol, vmask, y, ym) in enumerate(dl_tr):
            vol, vmask, y, ym = (t.cuda(non_blocking=True) for t in (vol, vmask, y, ym))
            with torch.autocast("cuda", dtype=torch.float16):
                loss = masked_bce(model(vol, vmask).float(), y, ym)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
            if it % 100 == 0:
                print(f"  ep{ep} it{it}/{len(dl_tr)} loss {run/(it+1):.4f} "
                      f"({(time.time()-t0)/(it+1):.2f} s/it)", flush=True)

        p_va = predict(model, dl_va)
        va_auc, _ = mean_auc(va[LABELS].values.astype(np.float32), p_va)
        va_ref_auc, _ = mean_auc(y_va_ref, p_va)
        gold_auc, _ = mean_auc(gold[LABELS].values.astype(np.float32), predict(model, dl_gold))
        log.append({"epoch": ep, "loss": run / len(dl_tr), "val_auc": va_auc,
                    "val_ref_auc": va_ref_auc, "gold_auc": gold_auc,
                    "min": (time.time() - t0) / 60})
        print(f"Epoch {ep}: loss {run/len(dl_tr):.4f} | val AUC {va_auc:.4f} | "
              f"val-ref AUC {va_ref_auc:.4f} | gold AUC {gold_auc:.4f} | "
              f"{(time.time()-t0)/60:.1f} menit", flush=True)
        # checkpoint dipilih dari val (label perbaikan), bukan gold
        if va_ref_auc > best:
            best = va_ref_auc
            torch.save(net.state_dict(), os.path.join(OUT, f"{tag}_best.pt"))
        pd.DataFrame(log).to_csv(os.path.join(OUT, f"train_log_{tag}.csv"), index=False)

    # -------- evaluasi akhir checkpoint terbaik pada 58 label emas --------
    net.load_state_dict(torch.load(os.path.join(OUT, f"{tag}_best.pt")))
    p_gold = predict(model, dl_gold)
    p_va = predict(model, dl_va)                                # untuk menentukan bobot ensemble nanti
    pd.concat([va[["StudyInstanceUID"]].reset_index(drop=True), pd.DataFrame(p_va, columns=LABELS)],
              axis=1).to_csv(os.path.join(OUT, f"val_preds_{tag}.csv"), index=False)
    g_auc, per = mean_auc(gold[LABELS].values.astype(np.float32), p_gold)
    print(f"\n=== [{tag}] Checkpoint terbaik (val-ref AUC {best:.4f}) pada label emas ===")
    for l, a in zip(LABELS, per):
        print(f"{l:18s} AUC={a:.3f}")
    print(f"Mean AUC gold: {g_auc:.4f}")
    pd.concat([gold[["StudyInstanceUID"]], pd.DataFrame(p_gold, columns=LABELS)], axis=1) \
      .to_csv(os.path.join(OUT, f"gold_preds_{tag}.csv"), index=False)
    os._exit(0)   # hindari proses DataLoader yang menggantung saat keluar


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--backbone", default=CFG["backbone"])
    ap.add_argument("--tag", default=CFG["tag"])
    ap.add_argument("--epochs", type=int, default=CFG["epochs"])
    ap.add_argument("--lr_backbone", type=float, default=CFG["lr_backbone"])
    ap.add_argument("--lr_head", type=float, default=CFG["lr_head"])
    ap.add_argument("--batch", type=int, default=CFG["batch"])
    ap.add_argument("--llrd", type=float, default=CFG["llrd"])
    ap.add_argument("--seed", type=int, default=CFG["seed"])
    ap.add_argument("--fold", type=int, default=CFG["fold"])
    ap.add_argument("--nfolds", type=int, default=CFG["nfolds"])
    ap.add_argument("--alpha", type=float, default=CFG["alpha"])
    ap.add_argument("--oof", default=CFG["oof"])
    ap.add_argument("--meta", default=CFG["meta"])
    ap.add_argument("--debug", action="store_true")
    CFG.update(vars(ap.parse_args()))
    main()

Writing /kaggle/working/08_train_student.py


In [2]:
%%bash
DEBUG=""
cd /kaggle/working
ARGS="--backbone efficientnet_b0 --lr_backbone 1e-4 --epochs 6 --alpha 0.5"

CUDA_VISIBLE_DEVICES=0 python -u 08_train_student.py $ARGS \
    --tag effb0_student_v1 --meta _meta.csv --batch 4 $DEBUG \
    > log_student_v1.txt 2>&1 &

CUDA_VISIBLE_DEVICES=1 python -u 08_train_student.py $ARGS \
    --tag effb0_student_v2 --meta _meta_v2.csv --batch 2 $DEBUG \
    > log_student_v2.txt 2>&1 &

wait
echo "===== Murid v1 (3 view, 224) ====="; tail -n 20 log_student_v1.txt
echo "===== Murid v2 (5 view, 256) ====="; tail -n 20 log_student_v2.txt

===== Murid v1 (3 view, 224) =====
  ep5 it600/978 loss 0.3385 (0.47 s/it)
  ep5 it700/978 loss 0.3362 (0.47 s/it)
  ep5 it800/978 loss 0.3357 (0.47 s/it)
  ep5 it900/978 loss 0.3352 (0.47 s/it)
Epoch 5: loss 0.3346 | val AUC 0.8355 | val-ref AUC 0.8364 | gold AUC 0.8182 | 7.9 menit

=== [effb0_student_v1] Checkpoint terbaik (val-ref AUC 0.8364) pada label emas ===
ACL                AUC=0.833
MCL                AUC=0.803
Medial Meniscus    AUC=0.747
Lateral Meniscus   AUC=0.702
Medial OA          AUC=0.981
Lateral OA         AUC=0.725
PF OA              AUC=0.768
Effusion           AUC=0.837
Synovitis          AUC=0.754
Baker's            AUC=0.951
Contusion          AUC=0.842
Fracture           AUC=0.874
Mean AUC gold: 0.8182
===== Murid v2 (5 view, 256) =====
  ep5 it1600/1957 loss 0.3242 (0.50 s/it)
  ep5 it1700/1957 loss 0.3239 (0.50 s/it)
  ep5 it1800/1957 loss 0.3237 (0.50 s/it)
  ep5 it1900/1957 loss 0.3242 (0.50 s/it)
Epoch 5: loss 0.3244 | val AUC 0.8440 | val-ref AUC 0.8446 

In [3]:
import pandas as pd
for tag in ["effb0_student_v1", "effb0_student_v2"]:
    try:
        print(f"\n===== {tag} =====")
        print(pd.read_csv(f"/kaggle/working/train_log_{tag}.csv").round(4).to_string(index=False))
    except FileNotFoundError:
        print("log belum ada")


===== effb0_student_v1 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc    min
     0 0.4491   0.7842       0.7853    0.8127 9.9937
     1 0.3874   0.8124       0.8138    0.8214 7.8722
     2 0.3657   0.8234       0.8243    0.8158 8.0746
     3 0.3512   0.8320       0.8326    0.8202 7.9419
     4 0.3404   0.8354       0.8363    0.8171 7.9353
     5 0.3346   0.8355       0.8364    0.8182 7.9025

===== effb0_student_v2 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.4478   0.7914       0.7920    0.8001 18.8976
     1 0.3893   0.8174       0.8181    0.8186 17.0575
     2 0.3653   0.8339       0.8349    0.8318 16.9346
     3 0.3464   0.8397       0.8404    0.8377 17.0232
     4 0.3321   0.8436       0.8443    0.8370 17.0648
     5 0.3244   0.8440       0.8446    0.8351 16.9796


In [4]:
!grep -H "Label diperbaiki\|Label  :\|Volume :" /kaggle/working/log_student_v*.txt

/kaggle/working/log_student_v1.txt:Volume : /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/vol_train
/kaggle/working/log_student_v1.txt:Label  : /kaggle/input/datasets/kellenvalerie/rsna-knee-report-labels/report_labels.csv
/kaggle/working/log_student_v1.txt:Label diperbaiki: alpha=0.5 (guru: oof_effb0_kf.csv)
/kaggle/working/log_student_v2.txt:Volume : /kaggle/input/notebooks/kellenvalerie/rsna-knee-02b-preprocess-mri-v2/vol_train_v2
/kaggle/working/log_student_v2.txt:Label  : /kaggle/input/datasets/kellenvalerie/rsna-knee-report-labels/report_labels.csv
/kaggle/working/log_student_v2.txt:Label diperbaiki: alpha=0.5 (guru: oof_effb0_kf.csv)
